In [10]:
import pandas as pd
import numpy as np


In [11]:
def process_nhanes_cycle(path_dict, cycle_name):

    # Load files
    demo = pd.read_sas(path_dict["DEMO"])
    dpq = pd.read_sas(path_dict["DPQ"])
    bmx = pd.read_sas(path_dict["BMX"])
    glu = pd.read_sas(path_dict["GLU"])
    cbc = pd.read_sas(path_dict["CBC"])

    if "CRP" in path_dict:
        crp_df = pd.read_sas(path_dict["CRP"])
    elif "BIOPRO" in path_dict:
        crp_df = pd.read_sas(path_dict["BIOPRO"])
    else:
        raise ValueError("Provide either 'CRP' or 'BIOPRO' in path_dict")


    # Merge
    df = demo.copy()
    df = df.merge(dpq, on="SEQN", how="left")
    df = df.merge(bmx, on="SEQN", how="left")
    df = df.merge(crp_df, on="SEQN", how="left")
    df = df.merge(glu, on="SEQN", how="left")
    df = df.merge(cbc, on="SEQN", how="left")

    # Clean missing value codes
    df.replace([7, 77, 777, 9, 99, 999], np.nan, inplace=True)

    # Create PHQ-9 score
    phq_cols = [
        "DPQ010","DPQ020","DPQ030","DPQ040","DPQ050",
        "DPQ060","DPQ070","DPQ080","DPQ090"
    ]

    df["phq9_score"] = df[phq_cols].sum(axis=1)

    # Depression label
    df["depression_label"] = (df["phq9_score"] >= 10).astype(int)

     # Suicide item
    if "DPQ090" in df.columns:
        df["suicide_q9"] = df["DPQ090"]
    else:
        df["suicide_q9"] = np.nan
     # Robust CRP handling
    crp_source_col = None
    for c in ["LBXCRP", "LBXSCRP", "URXCRP"]:
        if c in df.columns:
            crp_source_col = c
            break

    # Feature selection (rename)
     # Rename main variables if present
    rename_map = {
        "RIDAGEYR": "age",
        "RIAGENDR": "sex",
        "RIDRETH1": "race",
        "DMDEDUC2": "education",
        "DMDMARTL": "marital_status",
        "INDFMPIR": "income_ratio",
        "BMXWT": "weight",
        "BMXHT": "height",
        "BMXBMI": "bmi",
        "BMXWAIST": "waist",
        "BMXHIP": "hip",
        "LBXGLU": "glucose",
        "LBXWBCSI": "wbc",
        "LBXHGB": "hemoglobin",
        "LBXPLTSI": "platelet"
    }
    if crp_source_col is not None:
        rename_map[crp_source_col] = "crp"

    df = df.rename(columns=rename_map)

    # Waist-hip ratio
    if "waist" in df.columns and "hip" in df.columns:
        df["waist_hip_ratio"] = df["waist"] / df["hip"]

    # Keep only important columns
    keep_cols = [
        "SEQN","age","sex","race","education","marital_status","income_ratio",
        "weight","height","bmi","waist","hip","waist_hip_ratio",
        "glucose","crp","wbc","hemoglobin","platelet",
        "phq9_score","depression_label","suicide_q9"
    ]

    df = df[[col for col in keep_cols if col in df.columns]]

    # Drop bad columns (too much missing)
    df = df.loc[:, df.isna().mean() < 0.5]

    # Drop rows without target
    df = df.dropna(subset=["phq9_score"])

    # Add cycle column
    df["cycle"] = cycle_name

    print(f"{cycle_name} shape:", df.shape)

    return df

In [12]:
paths_2007 = {
    "DEMO": "DEMO_E.xpt",
    "DPQ":  "DPQ_E.xpt",
    "BMX":  "BMX_E.xpt",
    "CRP":  "CRP_E.xpt",
    "GLU":  "GLU_E.xpt",
    "CBC":  "CBC_E.xpt"
}
paths_2009 = {
    "DEMO": "DEMO_F.xpt",
    "DPQ":  "DPQ_F.xpt",
    "BMX":  "BMX_F.xpt",
    "CRP":  "CRP_F.xpt",
    "GLU":  "GLU_F.xpt",
    "CBC":  "CBC_F.xpt"
}
paths_2011 = {
    "DEMO": "DEMO_G.xpt",
    "DPQ":  "DPQ_G.xpt",
    "BMX":  "BMX_G.xpt",
    "GLU":  "GLU_G.xpt",
    "CBC":  "CBC_G.xpt",
    "BIOPRO": "BIOPRO_G.xpt"
}
paths_2013 = {
    "DEMO": "DEMO_H.xpt",
    "DPQ":  "DPQ_H.xpt",
    "BMX":  "BMX_H.xpt",
    "GLU":  "GLU_H.xpt",
    "CBC":  "CBC_H.xpt",
    "BIOPRO": "BIOPRO_H.xpt"
}
df_2007 = process_nhanes_cycle(paths_2007, "2007_2008")
df_2009 = process_nhanes_cycle(paths_2009, "2009_2010")
df_2011 = process_nhanes_cycle(paths_2011, "2011_2012")
df_2013 = process_nhanes_cycle(paths_2013, "2013_2014")

2007_2008 shape: (10149, 19)
2009_2010 shape: (10537, 19)
2011_2012 shape: (9756, 18)
2013_2014 shape: (10175, 18)


In [13]:
final_df = pd.concat([
    df_2007,
    df_2009,
    df_2011,
    df_2013
], ignore_index=True)

print(final_df.shape)
print(final_df["cycle"].value_counts())

(40617, 19)
cycle
2009_2010    10537
2013_2014    10175
2007_2008    10149
2011_2012     9756
Name: count, dtype: int64


In [14]:
final_df.head()
final_df.columns
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40617 entries, 0 to 40616
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   SEQN              40617 non-null  float64
 1   age               38704 non-null  float64
 2   sex               40617 non-null  float64
 3   race              40617 non-null  float64
 4   education         23448 non-null  float64
 5   marital_status    23464 non-null  float64
 6   income_ratio      37102 non-null  float64
 7   weight            38621 non-null  float64
 8   height            35966 non-null  float64
 9   bmi               35930 non-null  float64
 10  waist             34208 non-null  float64
 11  crp               15107 non-null  float64
 12  wbc               32969 non-null  float64
 13  hemoglobin        33560 non-null  float64
 14  platelet          33556 non-null  float64
 15  phq9_score        40617 non-null  float64
 16  depression_label  40617 non-null  int64 

In [15]:
final_df.isna().mean().sort_values(ascending=False)

,0
crp,0.628062
suicide_q9,0.474777
education,0.422705
marital_status,0.422311
wbc,0.188296
platelet,0.173843
hemoglobin,0.173745
waist,0.157791
bmi,0.115395
height,0.114509


In [ ]:
final_df["depression_label"].value_counts(normalize=True)

,proportion
depression_label,
0,0.950415
1,0.049585


In [16]:
final_df["phq9_score"].describe()

,phq9_score
count,4.061700e+04
mean,1.731590e+00
std,3.579659e+00
min,0.000000e+00
25%,0.000000e+00
50%,4.857845e-78
75%,2.000000e+00
max,2.700000e+01


In [17]:
final_df.groupby("sex")["depression_label"].mean()

,depression_label
sex,
1.0,0.035084
2.0,0.063904


what affects depression most

In [18]:
final_df.corr(numeric_only=True)["phq9_score"].sort_values(ascending=False)

,phq9_score
phq9_score,1.000000
depression_label,0.797157
suicide_q9,0.442999
weight,0.315462
waist,0.312798
bmi,0.288432
age,0.284705
height,0.225389
crp,0.109044
marital_status,0.094240


In [19]:
final_df.columns

Index(['SEQN', 'age', 'sex', 'race', 'education', 'marital_status',
       'income_ratio', 'weight', 'height', 'bmi', 'waist', 'crp', 'wbc',
       'hemoglobin', 'platelet', 'phq9_score', 'depression_label',
       'suicide_q9', 'cycle'],
      dtype='object')

**Normal logistic regression**

In [20]:
features = [col for col in [
    "age","sex","income_ratio",
    "bmi","waist_hip_ratio",
    "crp","wbc","glucose"
] if col in final_df.columns]

print("Using features:", features)

X = final_df[features]

# safer fill
X = X.fillna(X.median())

y = final_df["depression_label"]

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Using features: ['age', 'sex', 'income_ratio', 'bmi', 'crp', 'wbc']
Accuracy: 0.9556868537666174


logistic regression showed high accuracy but failed to detect depressed individuals due to class imbalance.

In [21]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[7764    7]
 [ 353    0]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      7771
           1       0.00      0.00      0.00       353

    accuracy                           0.96      8124
   macro avg       0.48      0.50      0.49      8124
weighted avg       0.91      0.96      0.93      8124



In [22]:
from sklearn.linear_model import LogisticRegression

model_balanced = LogisticRegression(max_iter=1000, class_weight="balanced")
model_balanced.fit(X_train, y_train)

y_pred_balanced = model_balanced.predict(X_test)

from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, y_pred_balanced))
print(classification_report(y_test, y_pred_balanced))

[[5386 2385]
 [ 104  249]]
              precision    recall  f1-score   support

           0       0.98      0.69      0.81      7771
           1       0.09      0.71      0.17       353

    accuracy                           0.69      8124
   macro avg       0.54      0.70      0.49      8124
weighted avg       0.94      0.69      0.78      8124



After applying class balancing, recall for depression improved significantly, indicating better identification of depressed participants, although precision decreased.

false positives are more

In [23]:
X = final_df[features].fillna(final_df[features].mean())
y = final_df["depression_label"]

In [24]:
import pandas as pd

importance = pd.Series(model.coef_[0], index=features)
print(importance.sort_values(ascending=False))

sex             0.479984
bmi             0.062157
crp             0.059720
wbc             0.036363
age             0.022690
income_ratio   -0.342846
dtype: float64


In [25]:
final_df.isna().mean().sort_values(ascending=False)
# overall depression %
final_df["depression_label"].mean()

# by gender
final_df.groupby("sex")["depression_label"].mean()

# by age
final_df.groupby("age")["depression_label"].mean().head(20)

# PHQ score
final_df["phq9_score"].describe()

,phq9_score
count,4.061700e+04
mean,1.731590e+00
std,3.579659e+00
min,0.000000e+00
25%,0.000000e+00
50%,4.857845e-78
75%,2.000000e+00
max,2.700000e+01


In [26]:
print(final_df.shape)
print(final_df.columns)

(40617, 19)
Index(['SEQN', 'age', 'sex', 'race', 'education', 'marital_status',
       'income_ratio', 'weight', 'height', 'bmi', 'waist', 'crp', 'wbc',
       'hemoglobin', 'platelet', 'phq9_score', 'depression_label',
       'suicide_q9', 'cycle'],
      dtype='object')


runing model without CRP

In [27]:
features_no_crp = [col for col in [
    "age", "sex", "income_ratio", "bmi", "wbc"
] if col in final_df.columns]

print("Using features:", features_no_crp)

X2 = final_df[features_no_crp].fillna(final_df[features_no_crp].median())
y2 = final_df["depression_label"]

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2)

model_balanced_2 = LogisticRegression(max_iter=1000, class_weight="balanced")
model_balanced_2.fit(X_train2, y_train2)

y_pred_balanced_2 = model_balanced_2.predict(X_test2)

print(confusion_matrix(y_test2, y_pred_balanced_2))
print(classification_report(y_test2, y_pred_balanced_2))

Using features: ['age', 'sex', 'income_ratio', 'bmi', 'wbc']
[[5322 2405]
 [ 105  292]]
              precision    recall  f1-score   support

           0       0.98      0.69      0.81      7727
           1       0.11      0.74      0.19       397

    accuracy                           0.69      8124
   macro avg       0.54      0.71      0.50      8124
weighted avg       0.94      0.69      0.78      8124



In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

features_reg = [col for col in [
    "age", "sex", "income_ratio", "bmi", "wbc"
] if col in final_df.columns]

X_reg = final_df[features_reg].fillna(final_df[features_reg].median())
y_reg = final_df["phq9_score"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2)

reg_model = LinearRegression()
reg_model.fit(X_train_r, y_train_r)

y_pred_r = reg_model.predict(X_test_r)

print("MAE:", mean_absolute_error(y_test_r, y_pred_r))
print("R2:", r2_score(y_test_r, y_pred_r))

MAE: 2.0429271790209245
R2: 0.12994047284801935


**CRP improves detection of depression despite high missingness.**



*   Dataset combined across multiple NHANES cycles (~30k samples)
*   Depression prevalence ~5%
*   Strong class imbalance observed
*   Standard logistic regression failed to detect depression
*   Balanced logistic regression significantly improved recall
*   CRP contributed to better detection performance
*   Regression model achieved MAE ≈ 2.0 but low R², indicating limited explanatory power
*   Additional features are needed to improve prediction

**Now i am adding more features.**

*   marital_status
*   education
*   race

In [29]:
features = [col for col in [
    "age", "sex", "income_ratio",
    "bmi", "wbc", "crp",
    "education", "marital_status", "race"
] if col in final_df.columns]

In [34]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[7764    7]
 [ 353    0]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      7771
           1       0.00      0.00      0.00       353

    accuracy                           0.96      8124
   macro avg       0.48      0.50      0.49      8124
weighted avg       0.91      0.96      0.93      8124



In [32]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_sm, y_train_sm)

y_pred_rf = rf.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      0.94      0.95      7771
           1       0.13      0.18      0.15       353

    accuracy                           0.91      8124
   macro avg       0.55      0.56      0.55      8124
weighted avg       0.93      0.91      0.92      8124



In [33]:
y_probs = rf.predict_proba(X_test)[:,1]

# lower threshold
y_pred_custom = (y_probs > 0.3).astype(int)

print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.97      0.85      0.91      7771
           1       0.11      0.42      0.18       353

    accuracy                           0.83      8124
   macro avg       0.54      0.63      0.54      8124
weighted avg       0.93      0.83      0.88      8124



in all above Balanced Logistic Regression models, only without CRP is performing well